In [2]:
# ============================================================
# 00 — CONNECT GOOGLE DRIVE / PROJECT
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ============================================================
# PROJECT PATH
# ============================================================

from pathlib import Path
import sys

PROJECT_DIR = Path(
    "/content/drive/MyDrive/FitnessML_Master"
)

print(f"Project directory: {PROJECT_DIR}")
print(f"Exists: {PROJECT_DIR.exists()}")

Project directory: /content/drive/MyDrive/FitnessML_Master
Exists: True


In [4]:
# ============================================================
# ADD PROJECT TO PYTHON PATH
# ============================================================

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project added to Python path.")

Project added to Python path.


In [5]:
# ============================================================
# 004 — FEATURE ENGINEERING
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import config_fitbit as config

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("=" * 70)
print("FITBIT FEATURE ENGINEERING")
print("=" * 70)

processed_path = (
    Path(config.RAW_DATA_DIR).parents[1]
    / "processed"
    / "fitbit_daily_processed.csv"
)

daily = pd.read_csv(processed_path)

daily["ActivityDate"] = pd.to_datetime(
    daily["ActivityDate"]
)

daily = (
    daily
    .sort_values(["Id", "ActivityDate"])
    .reset_index(drop=True)
)

print(f"Dataset shape: {daily.shape}")
print(f"Users: {daily['Id'].nunique()}")
print(
    f"Date range: "
    f"{daily['ActivityDate'].min().date()} → "
    f"{daily['ActivityDate'].max().date()}"
)

display(daily.head())

FITBIT FEATURE ENGINEERING
Dataset shape: (940, 15)
Users: 33
Date range: 2016-04-12 → 2016-05-12


,Id,ActivityDate,TotalSteps,TotalDistance,TrackerDistance,LoggedActivitiesDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories
0,1503960366,2016-04-12,13162,8.50,8.50,0.0,1.88,0.55,6.06,0.0,25,13,328,728,1985
1,1503960366,2016-04-13,10735,6.97,6.97,0.0,1.57,0.69,4.71,0.0,21,19,217,776,1797
2,1503960366,2016-04-14,10460,6.74,6.74,0.0,2.44,0.40,3.91,0.0,30,11,181,1218,1776
3,1503960366,2016-04-15,9762,6.28,6.28,0.0,2.14,1.26,2.83,0.0,29,34,209,726,1745
4,1503960366,2016-04-16,12669,8.16,8.16,0.0,2.71,0.41,5.04,0.0,36,10,221,773,1863


In [6]:
# ============================================================
# NEXT-DAY TARGET
# ============================================================

print("=" * 70)
print("NEXT-DAY TARGET")
print("=" * 70)

daily["target_calories_next_day"] = (
    daily
    .groupby("Id")["Calories"]
    .shift(-1)
)

print(
    "Target definition: "
    "Calories(t+1) from information available at day t"
)

display(
    daily[
        [
            "Id",
            "ActivityDate",
            "Calories",
            "target_calories_next_day"
        ]
    ].head(10)
)

NEXT-DAY TARGET
Target definition: Calories(t+1) from information available at day t


,Id,ActivityDate,Calories,target_calories_next_day
0,1503960366,2016-04-12,1985,1797.0
1,1503960366,2016-04-13,1797,1776.0
2,1503960366,2016-04-14,1776,1745.0
3,1503960366,2016-04-15,1745,1863.0
4,1503960366,2016-04-16,1863,1728.0
5,1503960366,2016-04-17,1728,1921.0
6,1503960366,2016-04-18,1921,2035.0
7,1503960366,2016-04-19,2035,1786.0
8,1503960366,2016-04-20,1786,1775.0
9,1503960366,2016-04-21,1775,1827.0


In [7]:
# ============================================================
# LAG FEATURES
# ============================================================

lag_columns = [
    "Calories",
    "TotalSteps",
    "TotalDistance",
    "VeryActiveMinutes",
    "FairlyActiveMinutes",
    "LightlyActiveMinutes",
    "SedentaryMinutes"
]

lags = [1, 2, 3, 7]

for column in lag_columns:

    for lag in lags:

        daily[f"{column}_lag_{lag}"] = (
            daily
            .groupby("Id")[column]
            .shift(lag)
        )

print("=" * 70)
print("LAG FEATURES CREATED")
print("=" * 70)

lag_feature_names = [
    column
    for column in daily.columns
    if "_lag_" in column
]

print(f"Lag features: {len(lag_feature_names)}")

display(
    daily[
        [
            "Id",
            "ActivityDate",
            "Calories",
            "Calories_lag_1",
            "Calories_lag_2",
            "TotalSteps_lag_1",
            "TotalSteps_lag_7"
        ]
    ].head(10)
)

LAG FEATURES CREATED
Lag features: 28


,Id,ActivityDate,Calories,Calories_lag_1,Calories_lag_2,TotalSteps_lag_1,TotalSteps_lag_7
0,1503960366,2016-04-12,1985,NaN,NaN,NaN,NaN
1,1503960366,2016-04-13,1797,1985.0,NaN,13162.0,NaN
2,1503960366,2016-04-14,1776,1797.0,1985.0,10735.0,NaN
3,1503960366,2016-04-15,1745,1776.0,1797.0,10460.0,NaN
4,1503960366,2016-04-16,1863,1745.0,1776.0,9762.0,NaN
5,1503960366,2016-04-17,1728,1863.0,1745.0,12669.0,NaN
6,1503960366,2016-04-18,1921,1728.0,1863.0,9705.0,NaN
7,1503960366,2016-04-19,2035,1921.0,1728.0,13019.0,13162.0
8,1503960366,2016-04-20,1786,2035.0,1921.0,15506.0,10735.0
9,1503960366,2016-04-21,1775,1786.0,2035.0,10544.0,10460.0


In [8]:
# ============================================================
# ROLLING FEATURES
# ============================================================

rolling_columns = [
    "Calories",
    "TotalSteps",
    "TotalDistance",
    "VeryActiveMinutes",
    "LightlyActiveMinutes",
    "SedentaryMinutes"
]

windows = [3, 7]

for column in rolling_columns:

    grouped = daily.groupby("Id")[column]

    for window in windows:

        daily[f"{column}_rolling_mean_{window}"] = (
            grouped
            .shift(1)
            .rolling(window)
            .mean()
            .reset_index(level=0, drop=True)
        )

        daily[f"{column}_rolling_std_{window}"] = (
            grouped
            .shift(1)
            .rolling(window)
            .std()
            .reset_index(level=0, drop=True)
        )

print("=" * 70)
print("ROLLING FEATURES CREATED")
print("=" * 70)

rolling_feature_names = [
    column
    for column in daily.columns
    if "rolling_" in column
]

print(f"Rolling features: {len(rolling_feature_names)}")

ROLLING FEATURES CREATED
Rolling features: 24


In [15]:
# ============================================================
# REMOVE PREVIOUS ROLLING FEATURES
# ============================================================

old_rolling = [
    c for c in daily.columns
    if "rolling_" in c
]

daily.drop(
    columns=old_rolling,
    inplace=True
)

print(f"Removed old rolling features: {len(old_rolling)}")

Removed old rolling features: 24


In [16]:
# ============================================================
# CORRECT USER-SPECIFIC ROLLING FEATURES
# ============================================================

rolling_columns = [
    "Calories",
    "TotalSteps",
    "TotalDistance",
    "VeryActiveMinutes",
    "LightlyActiveMinutes",
    "SedentaryMinutes"
]

windows = [3, 7]

for column in rolling_columns:

    for window in windows:

        daily[f"{column}_rolling_mean_{window}"] = (
            daily
            .groupby("Id")[column]
            .transform(
                lambda s: s.shift(1)
                          .rolling(window)
                          .mean()
            )
        )

        daily[f"{column}_rolling_std_{window}"] = (
            daily
            .groupby("Id")[column]
            .transform(
                lambda s: s.shift(1)
                          .rolling(window)
                          .std()
            )
        )

print("=" * 70)
print("ROLLING FEATURES REBUILT")
print("=" * 70)

print(
    "Rolling features:",
    len([
        c for c in daily.columns
        if "rolling_" in c
    ])
)

ROLLING FEATURES REBUILT
Rolling features: 24


In [17]:
# ============================================================
# TRUE TEMPORAL LEAKAGE / HORIZON CHECK
# ============================================================

print("=" * 70)
print("TEMPORAL HORIZON CHECK")
print("=" * 70)

daily["next_date"] = (
    daily.groupby("Id")["ActivityDate"].shift(-1)
)

daily["days_to_target"] = (
    daily["next_date"] - daily["ActivityDate"]
).dt.days

# How many records really have the next calendar day?
next_day_ok = (
    daily["days_to_target"] == 1
)

print(
    f"Rows with true next-day target: "
    f"{next_day_ok.sum():,}"
)

print(
    f"Rows where next observation is NOT next calendar day: "
    f"{(~next_day_ok & daily['next_date'].notna()).sum():,}"
)

print(
    f"Last observation per user: "
    f"{daily['next_date'].isna().sum():,}"
)

display(
    daily[
        ~next_day_ok &
        daily["next_date"].notna()
    ][
        [
            "Id",
            "ActivityDate",
            "next_date",
            "days_to_target"
        ]
    ].head(20)
)

TEMPORAL HORIZON CHECK
Rows with true next-day target: 907
Rows where next observation is NOT next calendar day: 0
Last observation per user: 33


,Id,ActivityDate,next_date,days_to_target


In [18]:
# ============================================================
# KEEP ONLY TRUE NEXT-DAY TARGETS
# ============================================================

daily["target_calories_next_day"] = (
    daily.groupby("Id")["Calories"].shift(-1)
)

daily.loc[
    daily["days_to_target"] != 1,
    "target_calories_next_day"
] = np.nan

print(
    "Valid next-day targets:",
    daily["target_calories_next_day"].notna().sum()
)

Valid next-day targets: 907


In [19]:
# ============================================================
# FEATURE AVAILABILITY
# ============================================================

print("=" * 70)
print("FEATURE AVAILABILITY")
print("=" * 70)

feature_columns = (
    lag_feature_names +
    rolling_feature_names
)

availability = pd.DataFrame({
    "missing": daily[feature_columns].isna().sum(),
    "missing_%": (
        daily[feature_columns].isna().mean() * 100
    )
})

display(
    availability
    .sort_values("missing_%", ascending=False)
    .head(20)
)

FEATURE AVAILABILITY


,missing,missing_%
Calories_lag_7,228,24.255319
FairlyActiveMinutes_lag_7,228,24.255319
TotalDistance_lag_7,228,24.255319
TotalSteps_lag_7,228,24.255319
VeryActiveMinutes_lag_7,228,24.255319
TotalDistance_rolling_mean_7,228,24.255319
VeryActiveMinutes_rolling_mean_7,228,24.255319
VeryActiveMinutes_rolling_std_7,228,24.255319
SedentaryMinutes_rolling_std_7,228,24.255319
SedentaryMinutes_rolling_mean_7,228,24.255319


In [28]:
# ============================================================
# FINAL MODELING DATASET
# ============================================================

base_features = [
    "TotalSteps",
    "TotalDistance",
    "TrackerDistance",
    "LoggedActivitiesDistance",
    "VeryActiveDistance",
    "ModeratelyActiveDistance",
    "LightActiveDistance",
    "SedentaryActiveDistance",
    "VeryActiveMinutes",
    "FairlyActiveMinutes",
    "LightlyActiveMinutes",
    "SedentaryMinutes",
    "Calories"
]

rolling_feature_names = [
    c for c in daily.columns
    if "rolling_" in c
]

feature_columns = (
    base_features +
    lag_feature_names +
    rolling_feature_names
)

model_columns = [
    "Id",
    "ActivityDate"
] + feature_columns + [
    "target_calories_next_day"
]

model_df = (
    daily[model_columns]
    .dropna()
    .reset_index(drop=True)
)

print("=" * 70)
print("FINAL MODELING DATASET")
print("=" * 70)

print(f"Rows: {len(model_df):,}")
print(f"Columns: {len(model_df.columns)}")
print(f"Users: {model_df['Id'].nunique()}")

print()
print(f"Base features: {len(base_features)}")
print(f"Lag features: {len(lag_feature_names)}")
print(f"Rolling features: {len(rolling_feature_names)}")

display(model_df.head())

FINAL MODELING DATASET
Rows: 680
Columns: 68
Users: 32

Base features: 13
Lag features: 28
Rolling features: 24


,Id,ActivityDate,TotalSteps,TotalDistance,TrackerDistance,LoggedActivitiesDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories,Calories_lag_1,Calories_lag_2,Calories_lag_3,Calories_lag_7,TotalSteps_lag_1,TotalSteps_lag_2,TotalSteps_lag_3,TotalSteps_lag_7,TotalDistance_lag_1,TotalDistance_lag_2,TotalDistance_lag_3,TotalDistance_lag_7,VeryActiveMinutes_lag_1,VeryActiveMinutes_lag_2,VeryActiveMinutes_lag_3,VeryActiveMinutes_lag_7,FairlyActiveMinutes_lag_1,FairlyActiveMinutes_lag_2,FairlyActiveMinutes_lag_3,FairlyActiveMinutes_lag_7,LightlyActiveMinutes_lag_1,LightlyActiveMinutes_lag_2,LightlyActiveMinutes_lag_3,LightlyActiveMinutes_lag_7,SedentaryMinutes_lag_1,SedentaryMinutes_lag_2,SedentaryMinutes_lag_3,SedentaryMinutes_lag_7,Calories_rolling_mean_3,Calories_rolling_std_3,Calories_rolling_mean_7,Calories_rolling_std_7,TotalSteps_rolling_mean_3,TotalSteps_rolling_std_3,TotalSteps_rolling_mean_7,TotalSteps_rolling_std_7,TotalDistance_rolling_mean_3,TotalDistance_rolling_std_3,TotalDistance_rolling_mean_7,TotalDistance_rolling_std_7,VeryActiveMinutes_rolling_mean_3,VeryActiveMinutes_rolling_std_3,VeryActiveMinutes_rolling_mean_7,VeryActiveMinutes_rolling_std_7,LightlyActiveMinutes_rolling_mean_3,LightlyActiveMinutes_rolling_std_3,LightlyActiveMinutes_rolling_mean_7,LightlyActiveMinutes_rolling_std_7,SedentaryMinutes_rolling_mean_3,SedentaryMinutes_rolling_std_3,SedentaryMinutes_rolling_mean_7,SedentaryMinutes_rolling_std_7,target_calories_next_day
0,1503960366,2016-04-19,15506,9.88,9.88,0.0,3.53,1.32,5.03,0.0,50,31,264,775,2035,1921.0,1728.0,1863.0,1985.0,13019.0,9705.0,12669.0,13162.0,8.59,6.48,8.16,8.50,42.0,38.0,36.0,25.0,16.0,20.0,10.0,13.0,233.0,164.0,221.0,328.0,1149.0,539.0,773.0,728.0,1837.333333,99.026932,1830.714286,95.764841,11797.666667,1820.732087,11358.857143,1538.733833,7.743333,1.115004,7.388571,0.993654,38.666667,3.055050,31.571429,7.457818,206.000000,36.864617,221.857143,52.594133,820.333333,307.742316,844.142857,245.957604,1786.0
1,1503960366,2016-04-20,10544,6.68,6.68,0.0,1.96,0.48,4.24,0.0,28,12,205,818,1786,2035.0,1921.0,1728.0,1797.0,15506.0,13019.0,9705.0,10735.0,9.88,8.59,6.48,6.97,50.0,42.0,38.0,21.0,31.0,16.0,20.0,19.0,264.0,233.0,164.0,217.0,775.0,1149.0,539.0,776.0,1894.666667,155.184836,1837.857143,109.997619,12743.333333,2910.308288,11693.714286,2135.758079,8.316667,1.716401,7.585714,1.330662,43.333333,6.110101,35.142857,9.494359,220.333333,51.189192,212.714286,32.968239,821.000000,307.590637,850.857143,242.880923,1775.0
2,1503960366,2016-04-21,9819,6.34,6.34,0.0,1.34,0.35,4.65,0.0,19,8,211,838,1775,1786.0,2035.0,1921.0,1776.0,10544.0,15506.0,13019.0,10460.0,6.68,9.88,8.59,6.74,28.0,50.0,42.0,30.0,12.0,31.0,16.0,11.0,205.0,264.0,233.0,181.0,818.0,775.0,1149.0,1218.0,1914.000000,124.647503,1836.285714,110.754555,13023.000000,2481.002418,11666.428571,2151.211818,8.383333,1.609979,7.544286,1.357275,40.000000,11.135529,36.142857,8.008924,234.000000,29.512709,211.000000,33.020196,914.000000,204.648479,856.857143,241.236695,1827.0
3,1503960366,2016-04-22,12764,8.13,8.13,0.0,4.76,1.12,2.24,0.0,66,27,130,1217,1827,1775.0,1786.0,2035.0,1745.0,9819.0,10544.0,15506.0,9762.0,6.34,6.68,9.88,6.28,19.0,28.0,50.0,29.0,8.0,12.0,31.0,34.0,211.0,205.0,264.0,209.0,838.0,818.0,775.0,726.0,1865.333333,147.038544,1836.142857,110.845882,11956.333333,3095.400836,11574.857143,2223.551920,7.633333,1.953083,7.487143,1.404383,32.333333,15.947832,34.571429,10.195704,226.666667,32.470499,215.285714,30.313442,810.333333,32.192132,802.571429,181.876201,1949.0
4,1503960366,2016-04-23,14371,9.04,9.04,0.0,2.81,0.87,5.36,0.0,41,21,262,732,1949,1827.0,1775.0,1786.0,1863.0,12764.0,9819.0,10544.0,12669.0,8.13,6.34,6.68,8.16,66.0,19.0,28.0,36.0,27.0,8.0,12.0,10.0,130.0,211.0,205.0,221.0,1217.0,838.0,818.0,773.0,1796.000000,27.404379,1847.857143,103.711826,11042.333333,1534.440723,12003.714286,2101.796035,7.050000,

In [21]:
user_counts = (
    model_df
    .groupby("Id")
    .size()
    .describe()
)

print("=" * 70)
print("OBSERVATIONS PER USER AFTER FEATURE ENGINEERING")
print("=" * 70)

display(user_counts)

OBSERVATIONS PER USER AFTER FEATURE ENGINEERING


,0
count,32.000000
mean,21.250000
std,3.618947
min,10.000000
25%,21.750000
50%,23.000000
75%,23.000000
max,23.000000


In [26]:
# ============================================================
# FINAL MODELING DATASET
# ============================================================

rolling_feature_names = [
    c for c in daily.columns
    if "rolling_" in c
]

feature_columns = (
    lag_feature_names +
    rolling_feature_names
)

model_columns = [
    "Id",
    "ActivityDate"
] + feature_columns + [
    "target_calories_next_day"
]

model_df = (
    daily[model_columns]
    .dropna()
    .reset_index(drop=True)
)

print("=" * 70)
print("FINAL MODELING DATASET")
print("=" * 70)

print(f"Rows: {len(model_df):,}")
print(f"Columns: {len(model_df.columns)}")
print(f"Users: {model_df['Id'].nunique()}")

display(model_df.head())

FINAL MODELING DATASET
Rows: 680
Columns: 55
Users: 32


,Id,ActivityDate,Calories_lag_1,Calories_lag_2,Calories_lag_3,Calories_lag_7,TotalSteps_lag_1,TotalSteps_lag_2,TotalSteps_lag_3,TotalSteps_lag_7,TotalDistance_lag_1,TotalDistance_lag_2,TotalDistance_lag_3,TotalDistance_lag_7,VeryActiveMinutes_lag_1,VeryActiveMinutes_lag_2,VeryActiveMinutes_lag_3,VeryActiveMinutes_lag_7,FairlyActiveMinutes_lag_1,FairlyActiveMinutes_lag_2,FairlyActiveMinutes_lag_3,FairlyActiveMinutes_lag_7,LightlyActiveMinutes_lag_1,LightlyActiveMinutes_lag_2,LightlyActiveMinutes_lag_3,LightlyActiveMinutes_lag_7,SedentaryMinutes_lag_1,SedentaryMinutes_lag_2,SedentaryMinutes_lag_3,SedentaryMinutes_lag_7,Calories_rolling_mean_3,Calories_rolling_std_3,Calories_rolling_mean_7,Calories_rolling_std_7,TotalSteps_rolling_mean_3,TotalSteps_rolling_std_3,TotalSteps_rolling_mean_7,TotalSteps_rolling_std_7,TotalDistance_rolling_mean_3,TotalDistance_rolling_std_3,TotalDistance_rolling_mean_7,TotalDistance_rolling_std_7,VeryActiveMinutes_rolling_mean_3,VeryActiveMinutes_rolling_std_3,VeryActiveMinutes_rolling_mean_7,VeryActiveMinutes_rolling_std_7,LightlyActiveMinutes_rolling_mean_3,LightlyActiveMinutes_rolling_std_3,LightlyActiveMinutes_rolling_mean_7,LightlyActiveMinutes_rolling_std_7,SedentaryMinutes_rolling_mean_3,SedentaryMinutes_rolling_std_3,SedentaryMinutes_rolling_mean_7,SedentaryMinutes_rolling_std_7,target_calories_next_day
0,1503960366,2016-04-19,1921.0,1728.0,1863.0,1985.0,13019.0,9705.0,12669.0,13162.0,8.59,6.48,8.16,8.50,42.0,38.0,36.0,25.0,16.0,20.0,10.0,13.0,233.0,164.0,221.0,328.0,1149.0,539.0,773.0,728.0,1837.333333,99.026932,1830.714286,95.764841,11797.666667,1820.732087,11358.857143,1538.733833,7.743333,1.115004,7.388571,0.993654,38.666667,3.055050,31.571429,7.457818,206.000000,36.864617,221.857143,52.594133,820.333333,307.742316,844.142857,245.957604,1786.0
1,1503960366,2016-04-20,2035.0,1921.0,1728.0,1797.0,15506.0,13019.0,9705.0,10735.0,9.88,8.59,6.48,6.97,50.0,42.0,38.0,21.0,31.0,16.0,20.0,19.0,264.0,233.0,164.0,217.0,775.0,1149.0,539.0,776.0,1894.666667,155.184836,1837.857143,109.997619,12743.333333,2910.308288,11693.714286,2135.758079,8.316667,1.716401,7.585714,1.330662,43.333333,6.110101,35.142857,9.494359,220.333333,51.189192,212.714286,32.968239,821.000000,307.590637,850.857143,242.880923,1775.0
2,1503960366,2016-04-21,1786.0,2035.0,1921.0,1776.0,10544.0,15506.0,13019.0,10460.0,6.68,9.88,8.59,6.74,28.0,50.0,42.0,30.0,12.0,31.0,16.0,11.0,205.0,264.0,233.0,181.0,818.0,775.0,1149.0,1218.0,1914.000000,124.647503,1836.285714,110.754555,13023.000000,2481.002418,11666.428571,2151.211818,8.383333,1.609979,7.544286,1.357275,40.000000,11.135529,36.142857,8.008924,234.000000,29.512709,211.000000,33.020196,914.000000,204.648479,856.857143,241.236695,1827.0
3,1503960366,2016-04-22,1775.0,1786.0,2035.0,1745.0,9819.0,10544.0,15506.0,9762.0,6.34,6.68,9.88,6.28,19.0,28.0,50.0,29.0,8.0,12.0,31.0,34.0,211.0,205.0,264.0,209.0,838.0,818.0,775.0,726.0,1865.333333,147.038544,1836.142857,110.845882,11956.333333,3095.400836,11574.857143,2223.551920,7.633333,1.953083,7.487143,1.404383,32.333333,15.947832,34.571429,10.195704,226.666667,32.470499,215.285714,30.313442,810.333333,32.192132,802.571429,181.876201,1949.0
4,1503960366,2016-04-23,1827.0,1775.0,1786.0,1863.0,12764.0,9819.0,10544.0,12669.0,8.13,6.34,6.68,8.16,66.0,19.0,28.0,36.0,27.0,8.0,12.0,10.0,130.0,211.0,205.0,221.0,1217.0,838.0,818.0,773.0,1796.000000,27.404379,1847.857143,103.711826,11042.333333,1534.440723,12003.714286,2101.796035,7.050000,0.950631,7.751429,1.310273,37.666667,24.946610,39.857143,15.192417,182.000000,45.133136,204.000000,44.452222,957.666667,224.811773,872.714286,234.492867,1788.0


In [24]:
# ============================================================
# FINAL TEMPORAL INTEGRITY CHECK
# ============================================================

print("=" * 70)
print("FINAL TEMPORAL INTEGRITY CHECK")
print("=" * 70)

# Expected values calculated directly from the original
# complete chronological dataset

expected_target = (
    daily.groupby("Id")["Calories"]
    .shift(-1)
)

expected_date = (
    daily.groupby("Id")["ActivityDate"]
    .shift(-1)
)

# Only rows that have a valid next-day target
valid = daily["target_calories_next_day"].notna()

# 1. Target value check
target_mismatches = (
    daily.loc[valid, "target_calories_next_day"]
    != expected_target.loc[valid]
).sum()

# 2. Target horizon check
horizon_days = (
    expected_date.loc[valid]
    - daily.loc[valid, "ActivityDate"]
).dt.days

invalid_horizons = (
    horizon_days != 1
).sum()

print(f"Target value mismatches: {target_mismatches}")
print(f"Invalid target horizons: {invalid_horizons}")

if target_mismatches == 0 and invalid_horizons == 0:
    print()
    print("✓ TEMPORAL INTEGRITY CHECK PASSED")
else:
    print()
    print("⚠ TEMPORAL INTEGRITY CHECK FAILED")

FINAL TEMPORAL INTEGRITY CHECK
Target value mismatches: 0
Invalid target horizons: 0

✓ TEMPORAL INTEGRITY CHECK PASSED


In [30]:
# ============================================================
# SAVE FEATURE-ENGINEERED MODELING DATASET
# ============================================================

processed_dir = Path(config.RAW_DATA_DIR).parents[1] / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

model_path = (
    processed_dir /
    "fitbit_modeling_features.csv"
)

model_df.to_csv(
    model_path,
    index=False
)

print("=" * 70)
print("FEATURE-ENGINEERED DATASET SAVED")
print("=" * 70)

print(f"Path: {model_path}")
print(f"Rows: {len(model_df):,}")
print(f"Columns: {len(model_df.columns)}")

FEATURE-ENGINEERED DATASET SAVED
Path: /content/drive/MyDrive/FitnessML_Master/data/processed/fitbit_modeling_features.csv
Rows: 680
Columns: 68
